In [1]:
import os
import numpy as np
import pandas as pd
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
#import statsmodels.api as sm
#from statsmodels.formula.api import ols
#from statsmodels.stats.anova import AnovaRM
#from sklearn.linear_model import LinearRegression
import csv

os.chdir('../../rf1-sra/stimuli/Scan-Investment_Game')


In [6]:
#Make a list of all the Trust Files
Trust_flist=[os.path.join(root, f) for root, dirs, files in os.walk('logs') for f in files if 'Trust-Ratings' in f]
Trust_flist
#Make a list of Dataframes
ratings_list=[]
for f in Trust_flist:
    sub=re.search('sub(.*)_',f).group(1)
    if any(i.isdigit() for i in sub):
        tmp_df=pd.read_csv(f)
        tmp_df['sub']=sub
        ratings_list.append(tmp_df)
#Concatonate the DataFrames together
tpr_df=pd.concat(ratings_list)
tpr_df=tpr_df.reset_index(drop=True)
#tpr_df['Rating'].astype(int)
tpr_df['Trait'].str.replace(' ', '').astype(int)


#Question 1, Trait 2 for each of the 3 is "Trustworthy" in the order of friend(3), stranger(2), computer(1) 
#Question 2, Trait 1 for each of the 3 is "Likeable" in the order of friend(3), stranger(2), computer(1)
#Question 3, Trait 0 for each of the 3 is "Approachable" in the order of friend(3), stranger(2), computer(1)

ValueError: cannot convert float NaN to integer

In [ ]:
Trait = {2:'Trustworthy', 1:'Likeable', 0:'Approachable'}
partner = {3:'Friend',2:'Stranger',1:'Computer'}
tpr_df['partner']=tpr_df['Partner'].map(partner)
tpr_df['Outcome']=tpr_df['Trait'].map(Trait)
tpr_df
#fig = sns.barplot(y='Rating',x='Outcome',hue='partner', data=tpr_df, palette=['tab:blue','orangered','gold'], edgecolor='black', linewidth = 1)
#plt.savefig('../../derivatives/SR_partner_ratings_anova.svg')
#plt.show()

In [ ]:
#This loop collapses the 6 rows of Shared Reward Partner Ratings into 1 row for each participant
collapsed = {}
for index, row in tpr_df.iterrows():
    if(not row['sub'] in collapsed):
        collapsed[row['sub']] = {}
    column_name = Trait[row['Trait']] + '-' + partner[row['Partner']]
    collapsed[row['sub']][column_name] = row['Rating']
#PR = Partner Ratings from post-scan task
PR = pd.DataFrame(collapsed).T
PR = PR.reset_index().rename(columns = {'index':'sub'})
PR